# 🔞 Task 3: Netflix Audience Rating Classification
### Multi-Class Classification: Predicting Content Age & Audience Ratings

---

## 1. Executive Summary & Problem Formulation
Audience rating systems (e.g., `TV-MA`, `TV-14`, `PG-13`, `R`, `TV-PG`, `TV-Y7`, `TV-Y`, `PG`, `TV-G`) protect viewers and comply with regional censorship laws.
Predicting audience ratings from descriptive content features (genres, duration, media type, release year) enables automated content categorization.

This task formulates a **Multi-Class Classification Problem**:
$$y \in \{ \text{TV-MA}, \text{TV-14}, \text{TV-PG}, \text{R}, \text{PG-13}, \text{TV-Y7}, \text{TV-Y}, \text{PG}, \text{TV-G} \}$$


In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from src.data_loader import get_preprocessed_data
from src.classifier import AudienceRatingClassifier

df = get_preprocessed_data('../data/Dataset.csv')
print("Audience Rating Distribution:")
print(df['rating'].value_counts().head(10))


Audience Rating Distribution:
rating
TV-MA    3205
TV-14    2157
TV-PG     861
R         799
PG-13     490
TV-Y7     333
TV-Y      306
PG        287
TV-G      220
NR         79
Name: count, dtype: int64


## 2. Multi-Class Model Training & Comparison
We evaluate:
- **Random Forest Classifier (Balanced Weights)**: Accounts for class imbalance across 9 rating categories.
- **Decision Tree Classifier**: Single interpretable tree baseline.


In [2]:
rf_rating = AudienceRatingClassifier(model_type='rf')
rf_rating_metrics = rf_rating.train(df)

dt_rating = AudienceRatingClassifier(model_type='dt')
dt_rating_metrics = dt_rating.train(df)

comp_df = pd.DataFrame([
    {"Model": "Random Forest", "Accuracy": rf_rating_metrics['accuracy'], "Weighted F1": rf_rating_metrics['weighted_f1'], "Macro F1": rf_rating_metrics['macro_f1']},
    {"Model": "Decision Tree", "Accuracy": dt_rating_metrics['accuracy'], "Weighted F1": dt_rating_metrics['weighted_f1'], "Macro F1": dt_rating_metrics['macro_f1']}
])
comp_df


Model  Accuracy  Weighted F1  Macro F1
0   Random Forest    0.4717       0.4821    0.4546
1   Decision Tree    0.4348       0.4480    0.4323


## 3. Classification Report & Per-Class Breakdown


In [3]:
print("Classification Report (Random Forest):")
rep = rf_rating_metrics['classification_report']
for cls_name in rf_rating_metrics['classes']:
    if cls_name in rep:
        stats = rep[cls_name]
        print(f"Class {cls_name:8s} | Precision: {stats['precision']:.3f} | Recall: {stats['recall']:.3f} | F1: {stats['f1-score']:.3f} | Support: {stats['support']}")


Classification Report (Random Forest):
Class PG       | Precision: 0.286 | Recall: 0.386 | F1: 0.328 | Support: 57
Class PG-13    | Precision: 0.237 | Recall: 0.235 | F1: 0.236 | Support: 98
Class R        | Precision: 0.442 | Recall: 0.506 | F1: 0.472 | Support: 160
Class TV-14    | Precision: 0.457 | Recall: 0.448 | F1: 0.452 | Support: 431
Class TV-G     | Precision: 0.231 | Recall: 0.341 | F1: 0.275 | Support: 44
Class TV-MA    | Precision: 0.598 | Recall: 0.536 | F1: 0.565 | Support: 641
Class TV-PG    | Precision: 0.239 | Recall: 0.302 | F1: 0.267 | Support: 172
Class TV-Y     | Precision: 0.638 | Recall: 0.721 | F1: 0.677 | Support: 61
Class TV-Y7    | Precision: 0.478 | Recall: 0.478 | F1: 0.478 | Support: 67


## 4. Real-Time Audience Rating Prediction


In [4]:
test_samples = [
    {"listed_in": "Kids' TV, Animation", "type": "TV Show", "duration": "1 Season", "release_year": 2021},
    {"listed_in": "Stand-Up Comedy", "type": "Movie", "duration": "65 min", "release_year": 2020},
    {"listed_in": "Horror Movies, Thrillers", "type": "Movie", "duration": "98 min", "release_year": 2018}
]

for sample in test_samples:
    res = rf_rating.predict(sample)
    top_3 = list(res['probabilities'].items())[:3]
    top_3_str = ", ".join([f"{k}: {v*100:.1f}%" for k, v in top_3])
    print(f"Content: {sample['listed_in']} ({sample['type']})")
    print(f" -> Predicted Rating: {res['prediction']} (Confidence: {res['confidence']*100:.1f}%)")
    print(f"    Probabilities: {top_3_str}\n")


Content: Kids' TV, Animation (TV Show)
 -> Predicted Rating: TV-Y7 (Confidence: 61.4%)
    Probabilities: TV-Y7: 61.4%, TV-Y: 28.2%, TV-PG: 8.1%

Content: Stand-Up Comedy (Movie)
 -> Predicted Rating: TV-MA (Confidence: 68.9%)
    Probabilities: TV-MA: 68.9%, TV-14: 18.5%, R: 7.2%

Content: Horror Movies, Thrillers (Movie)
 -> Predicted Rating: R (Confidence: 54.3%)
    Probabilities: R: 54.3%, TV-MA: 29.8%, PG-13: 11.2%


## 5. Insights & Findings
1. Highly specialized genres (e.g. `Kids' TV`, `Stand-Up Comedy`, `Horror Movies`) yield confident rating classifications (`TV-Y`, `TV-MA`, `R`).
2. General genres (`Dramas`, `Comedies`) exhibit broader distributions between `TV-14` and `TV-MA`, where natural language synopsis or dialogue transcript analysis would provide added discriminative power.
